In [1]:
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.0+cpu


In [2]:
output_dir = Path("../Outputs")

X_train = np.load(output_dir / "X_train.npy")
X_val = np.load(output_dir / "X_val.npy")

y_train = np.load(output_dir / "y_train.npy")
y_val = np.load(output_dir / "y_val.npy")

print("Training data:", X_train.shape)
print("Validation data:", X_val.shape)

Training data: (31647, 50)
Validation data: (6782, 50)


In [3]:
X_train_tensor = torch.tensor(
    X_train, dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train, dtype=torch.float32
).reshape(-1, 1)

X_val_tensor = torch.tensor(
    X_val, dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val, dtype=torch.float32
).reshape(-1, 1)

print("X_train tensor:", X_train_tensor.shape)
print("y_train tensor:", y_train_tensor.shape)

X_train tensor: torch.Size([31647, 50])
y_train tensor: torch.Size([31647, 1])


In [4]:
class BankMarketingMLP(nn.Module):

    def __init__(self, input_size):
        super(BankMarketingMLP, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


input_size = X_train.shape[1]

model = BankMarketingMLP(input_size)

print(model)

BankMarketingMLP(
  (network): Sequential(
    (0): Linear(in_features=50, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [5]:
total_params = 0

print("Model Parameters:")
print("-" * 50)

for name, parameter in model.named_parameters():

    params = parameter.numel()
    total_params += params

    print(f"{name}: {params} parameters")

print("-" * 50)
print("Total Trainable Parameters:", total_params)

Model Parameters:
--------------------------------------------------
network.0.weight: 3200 parameters
network.0.bias: 64 parameters
network.2.weight: 2048 parameters
network.2.bias: 32 parameters
network.4.weight: 32 parameters
network.4.bias: 1 parameters
--------------------------------------------------
Total Trainable Parameters: 5377


In [6]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss Function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning Rate:", optimizer.param_groups[0]["lr"])

Loss Function: BCEWithLogitsLoss()
Optimizer: Adam
Learning Rate: 0.001


In [7]:
model.eval()

with torch.no_grad():
    sample_input = X_train_tensor[:5]

    logits = model(sample_input)
    probabilities = torch.sigmoid(logits)

print("Input shape:", sample_input.shape)
print("Output shape:", logits.shape)

print("\nPredicted Probabilities:")
print(probabilities)

Input shape: torch.Size([5, 50])
Output shape: torch.Size([5, 1])

Predicted Probabilities:
tensor([[0.4830],
        [0.4804],
        [0.4867],
        [0.4904],
        [0.4619]])
